# 논문3 FFMC 산불발생확률 모형 정리

논문3: **캐나다 산불 기상지수를 이용한 산불발생확률모형 개발 - 강원도 지역 산불발생을 중심으로 -**

핵심 결론:

- 논문3은 산불 피해규모가 아니라 **산불 발생확률**을 예측합니다.
- 입력은 캐나다 산불기상지수 중 `FFMC`입니다.
- 10일 평균 FFMC를 4단계 Index로 바꾸고, 로지스틱 회귀식에 넣습니다.
- 논문 판별율은 약 `63.6%`입니다.
- 우리 강원도 날씨 데이터에 논문식을 적용해 임시 발생확률 파일을 만들었습니다.

## 1. 경로 정리

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('../..').resolve()
paper3_script_path = ROOT / 'jgy/modeling/prepare_paper3_ffmc_probability.py'
paper3_result_path = ROOT / 'data/modeling/paper3_ffmc_gangwon_occurrence_probability.csv'

print('논문3 실행 스크립트:', paper3_script_path)
print('존재 여부:', paper3_script_path.exists())
print()
print('논문3 임시 발생확률 결과:', paper3_result_path)
print('존재 여부:', paper3_result_path.exists())

## 2. 논문3 모델 설명

논문3은 강원도 지역의 기상자료로 `FFMC`를 계산한 뒤, 봄철 산불조심기간의 10일 평균 FFMC와 산불 발생 유무를 로지스틱 회귀로 연결했습니다.

논문식:

```text
logit(P) = -0.529 + 0.422 * Indexed_FFMC
P = exp(logit) / (1 + exp(logit))
```

Index 기준:

```text
0.00  ~ 58.00  -> 1
58.01 ~ 70.50  -> 2
70.51 ~ 80.00  -> 3
80.01 ~ 99.00  -> 4
```

Index가 높을수록 미세연료가 더 건조하고, 산불 발생확률이 높아지는 구조입니다.

## 3. 강원도 데이터 적용 결과

In [ ]:
paper3 = pd.read_csv(paper3_result_path, encoding='utf-8-sig')

print('행/열:', paper3.shape)
print('날짜 범위:', paper3['날짜'].min(), '~', paper3['날짜'].max())
paper3.head()

## 4. 발생확률 요약

In [ ]:
print('Indexed FFMC 분포')
print(paper3['indexed_ffmc'].value_counts().sort_index())

print('\n발생확률 요약')
print(paper3['paper3_occurrence_probability'].describe().round(4))

결과 요약:

```text
전체 행 수: 67,252
Index 1: 40,593행, 발생확률 0.4733
Index 2: 10,943행, 발생확률 0.5781
Index 3: 8,548행, 발생확률 0.6763
Index 4: 7,168행, 발생확률 0.7612
평균 발생확률: 0.5468
최대 발생확률: 0.7612
```

논문식은 `Indexed_FFMC` 4단계만 사용하기 때문에, 같은 Index 안에서는 발생확률이 같은 값으로 나옵니다.

## 5. 최고 발생확률 사례

In [ ]:
cols = [
    '기상셀ID', '날짜', '기후권역', '기후지형유형',
    'ffmc', 'ffmc_10day_mean', 'indexed_ffmc', 'paper3_occurrence_probability'
]

paper3.sort_values('paper3_occurrence_probability', ascending=False)[cols].head(15)

최고 발생확률은 `0.7612`이고, `Indexed_FFMC = 4`인 모든 날짜/기상셀에서 동일하게 나옵니다.

대표 사례:

```text
기상셀ID: YS_0071
날짜: 2021-02-28
기후권역: 영서
기후지형유형: 영서 내륙형
FFMC 10일 평균: 80.416960
Indexed FFMC: 4
임시 발생확률: 0.7612
```

## 6. 현재 한계

```text
1. 논문3의 원래 연구는 2002~2006년 5개 관측소 기반입니다.
   지금 적용한 데이터는 2020~2021년 강원도 기상셀 데이터입니다.

2. FFMC는 원래 정오 기준 기상값을 많이 쓰지만,
   현재는 일평균 기온, 일평균 습도, 일평균 풍속, 일강수량으로 계산했습니다.

3. 논문식은 Indexed FFMC만 사용하는 단순 로지스틱 모형입니다.
   따라서 지역별/날짜별 미세한 차이는 FFMC Index 4단계 안에서 사라집니다.

4. 결과는 정식 검증 모델이 아니라 임시 발생확률입니다.
```

그래도 논문1, 논문2보다 지금 데이터에 바로 적용하기 쉬운 형태입니다. 피해면적 타깃이 없어도 날씨 기반 발생확률을 만들 수 있기 때문입니다.